# 7.2 Transfer Learning with PyTorch — Exercise

After learning how to combine outputs of multiple models for the same task, we now use knowledge from a pretrained model to solve a related task. Specifically, **we apply transfer learning to classify the 10 object classes of the CIFAR-10 dataset: airplanes, automobiles, birds, cats, deer, dogs, frogs, horses, ships, and trucks.**

We train three CNN architectures: a small baseline model, a regularized model with data augmentation, and a ResNet50 pretrained on ImageNet. All data loading, preprocessing, model definition, training, and persistence in this notebook use PyTorch and Torchvision.

The computations can be expensive. A CUDA-capable NVIDIA GPU, Apple Silicon GPU, or a hosted notebook environment is recommended.

## PyTorch setup

Use the official [PyTorch installation selector](https://pytorch.org/get-started/locally/) to select a wheel for your operating system and compute platform. The available files are also listed in the [PyTorch wheel index](https://download.pytorch.org/whl/torch/). Install the remaining notebook dependencies if needed:

```python
%pip install matplotlib pandas tqdm
```

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import ResNet50_Weights, resnet50
from tqdm.auto import tqdm

torch.manual_seed(42)

In [ ]:
# Select exactly one compute device. CPU is the default.
device = torch.device('cpu')

# macOS with Apple Silicon:
# device = torch.device('mps')

# Windows/Linux with a CUDA-capable NVIDIA GPU
# (change 0 to select another GPU):
# device = torch.device('cuda:0')

print(f'Using device: {device}')

We start by defining reusable helper functions:

- `_process_history` plots training and validation loss and accuracy.
- `_run_epoch` performs one training or validation epoch.
- `_train_model` runs the complete training loop and records its history.
- `_save_trained_model` stores a PyTorch `state_dict` and the history.
- `_load_trained_model` restores them on the selected device.

In [ ]:
MODEL_DIR = Path.cwd() / 'trained_models'


def _process_history(history):
  result_frame = pd.DataFrame(history)
  fig, ax = plt.subplots(2, 1, sharex=True, figsize=(10, 8))

  ax[0].plot(
    result_frame[['loss', 'val_loss']],
    label=['Train Loss', 'Validation Loss'],
    linewidth=2,
    marker='o',
    markersize=3,
  )
  ax[0].set_ylabel('Loss')
  ax[0].legend(loc='upper left')

  ax[1].plot(
    result_frame[['accuracy', 'val_accuracy']],
    label=['Train Accuracy', 'Validation Accuracy'],
    linewidth=2,
    marker='o',
    markersize=3,
  )
  ax[1].set_ylabel('Accuracy')
  ax[1].set_xlabel('Epoch')
  ax[1].legend(loc='upper left')
  plt.subplots_adjust(hspace=0.05)
  plt.show()


def _run_epoch(model, data_loader, loss_fn, optimizer=None):
  is_training = optimizer is not None
  model.train(is_training)
  total_loss = 0.0
  total_correct = 0
  total_items = 0

  context = torch.enable_grad() if is_training else torch.inference_mode()
  with context:
    for inputs, targets in data_loader:
      inputs = inputs.to(device)
      targets = targets.to(device)

      if is_training:
        optimizer.zero_grad()

      logits = model(inputs)
      loss = loss_fn(logits, targets)

      if is_training:
        loss.backward()
        optimizer.step()

      batch_size = targets.size(0)
      total_loss += loss.item() * batch_size
      total_correct += (logits.argmax(dim=1) == targets).sum().item()
      total_items += batch_size

  return total_loss / total_items, total_correct / total_items


def _train_model(model, train_loader, validation_loader, loss_fn, optimizer, epochs):
  history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}

  for epoch in tqdm(range(epochs), desc='Epochs'):
    train_loss, train_accuracy = _run_epoch(
      model, train_loader, loss_fn, optimizer
    )
    validation_loss, validation_accuracy = _run_epoch(
      model, validation_loader, loss_fn
    )

    history['loss'].append(train_loss)
    history['val_loss'].append(validation_loss)
    history['accuracy'].append(train_accuracy)
    history['val_accuracy'].append(validation_accuracy)

    print(
      f'Epoch {epoch + 1:03d}/{epochs}: '
      f'loss={train_loss:.4f}, accuracy={train_accuracy:.4f}, '
      f'val_loss={validation_loss:.4f}, '
      f'val_accuracy={validation_accuracy:.4f}'
    )

  return history


def _save_trained_model(model, history, model_name):
  MODEL_DIR.mkdir(parents=True, exist_ok=True)
  with (MODEL_DIR / f'{model_name}_history.json').open('w') as file:
    json.dump(history, file)
  torch.save(model.state_dict(), MODEL_DIR / f'{model_name}.pt')


def _load_trained_model(model, model_name):
  with (MODEL_DIR / f'{model_name}_history.json').open('r') as file:
    history = json.load(file)

  model_path = MODEL_DIR / f'{model_name}.pt'
  if not model_path.exists():
    return history, None

  model.to(device)
  state_dict = torch.load(model_path, map_location=device, weights_only=True)
  model.load_state_dict(state_dict)
  model.eval()
  return history, model

## Data Loading and Preprocessing

Torchvision downloads CIFAR-10 and returns integer class labels. `CrossEntropyLoss` expects these class indices directly, so no categorical or one-hot encoding is required. The baseline uses only tensor conversion; augmentation for the other models is defined in their respective sections.

In [ ]:
DATA_DIR = next(
  (parent / 'data' for parent in (Path.cwd(), *Path.cwd().parents)
   if (parent / 'data').is_dir()),
  None,
)
if DATA_DIR is None:
  raise FileNotFoundError('Could not find the repository data directory.')
PIN_MEMORY = device.type == 'cuda'
PERSISTENT_WORKERS = device.type == 'cuda'
NUM_WORKERS = 4 if device.type == 'cuda' else 0

class_names = [
  'airplane', 'automobile', 'bird', 'cat', 'deer',
  'dog', 'frog', 'horse', 'ship', 'truck'
]

basic_transform = transforms.ToTensor()

train_data = datasets.CIFAR10(
  root=DATA_DIR, train=True, download=True, transform=basic_transform
)
test_data = datasets.CIFAR10(
  root=DATA_DIR, train=False, download=True, transform=basic_transform
)

print(f'Training samples: {len(train_data):,}')
print(f'Test samples: {len(test_data):,}')

In [ ]:
plt.figure(figsize=(10, 10))
for index in range(25):
  image, label = train_data[index]
  plt.subplot(5, 5, index + 1)
  plt.xticks([])
  plt.yticks([])
  plt.grid(False)
  plt.imshow(image.permute(1, 2, 0))
  plt.xlabel(class_names[label])
plt.show()

## Base Model

### **TASK: Base Model Training**

Create a small CNN trained directly on CIFAR-10. It serves as a baseline for understanding model behavior without prior knowledge.

1. Implement `BaseCNN` with two convolution and pooling stages:
   - `Conv2d` with 32 filters and kernel size 4, followed by ReLU and max pooling.
   - `Conv2d` with 64 filters and kernel size 3, followed by ReLU and max pooling.
   - Flatten the feature maps and add a dense layer with 256 units and ReLU.
   - Add an output layer for the 10 CIFAR-10 classes. Return logits; do not add softmax.
2. Instantiate the model on `device`.
3. Select an appropriate loss function and Adam optimizer.
4. Train the model with the provided helper, then save its weights and history.

In [ ]:
class BaseCNN(nn.Module):
  def __init__(self):
    super().__init__()

    # ---- YOUR CODE GOES HERE (START) ----
    self.features = nn.Sequential(
            
    )
        
    self.classifier = nn.Sequential(
            
    )
    # ---- YOUR CODE GOES HERE (END) ----

  def forward(self, inputs):
    # ---- YOUR CODE GOES HERE (START) ----
    return None
    # ---- YOUR CODE GOES HERE (END) ----

In [ ]:
BATCH_SIZE = 256
EPOCHS = 100

train_loader = DataLoader(
  train_data, batch_size=BATCH_SIZE, shuffle=True,
  num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT_WORKERS
)
test_loader = DataLoader(
  test_data, batch_size=BATCH_SIZE, shuffle=False,
  num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT_WORKERS
)

# ---- YOUR CODE GOES HERE (START) ----
# Instantiate BaseCNN, define the loss and optimizer, train the model, and save it as 'base_model_pytorch'.

base_model = None
loss_fn = None
optimizer = None

base_model_history = None

# ---- YOUR CODE GOES HERE (END) ----

_save_trained_model(base_model, base_model_history, 'base_model_pytorch')

### **TASK: Training Curve**

Plot the training curve and interpret it: Was the training successful? Did it stop at the right time?

In [ ]:
# ---- YOUR CODE GOES HERE (START) ----
base_model_history, base_model = None, None
# ---- YOUR CODE GOES HERE (END) ----

_process_history(base_model_history)

**Your interpretation:**

_Describe whether the baseline training was successful, identify possible overfitting or underfitting, and discuss whether training stopped at an appropriate epoch._

## Advanced Model

### **TASK: Advanced Model Training**

Create a stronger CNN with data augmentation and regularization:

1. Add random translation, horizontal flipping, and rotation to the training transform.
2. Build three convolutional blocks with 32, 64, and 128 channels. Each block contains two convolutional layers with padding, batch normalization, and ReLU, followed by max pooling.
3. Add dropout rates of 0.2, 0.3, and 0.4 after the respective pooling layers.
4. Add a dense layer with 128 units, batch normalization, ReLU, and dropout 0.5 before the output layer.
5. Apply He initialization to convolutional layers and the hidden dense layer.
6. Train and save the completed model.

In [ ]:
advanced_train_transform = transforms.Compose([
  # ---- YOUR CODE GOES HERE (START) ----
  # Add random translation, horizontal flipping, and rotation.
  # ---- YOUR CODE GOES HERE (END) ----
  transforms.ToTensor(),
])

advanced_train_data = datasets.CIFAR10(
  root=DATA_DIR, train=True, download=False,
  transform=advanced_train_transform
)

In [ ]:
class AdvancedCNN(nn.Module):
  def __init__(self):
    super().__init__()

    # ---- YOUR CODE GOES HERE (START) ----
    self.features = nn.Sequential(
                    
    )
                
    self.classifier = nn.Sequential(
                    
    )
    # ---- YOUR CODE GOES HERE (END) ----

  @staticmethod
  def _initialize_weights(module):
    if isinstance(module, nn.Conv2d):
      nn.init.kaiming_uniform_(module.weight, nonlinearity='relu')
      if module.bias is not None:
        nn.init.zeros_(module.bias)

  def forward(self, inputs):
    return self.classifier(self.features(inputs))

In [ ]:
BATCH_SIZE = 256
EPOCHS = 100

advanced_train_loader = DataLoader(
  advanced_train_data, batch_size=BATCH_SIZE, shuffle=True,
  num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT_WORKERS
)
advanced_test_loader = DataLoader(
  test_data, batch_size=BATCH_SIZE, shuffle=False,
  num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT_WORKERS
)

advanced_model = AdvancedCNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(advanced_model.parameters(), lr=1e-3)

advanced_model_history = _train_model(
  advanced_model, advanced_train_loader, advanced_test_loader,
  loss_fn, optimizer, EPOCHS
)
_save_trained_model(
  advanced_model, advanced_model_history, 'advanced_model_pytorch'
)

### **TASK: Training Curve**

Plot the training curve and interpret it: Was the training successful? Did it stop at the right time?

In [ ]:
advanced_model_history, advanced_model = _load_trained_model(
  AdvancedCNN(), 'advanced_model_pytorch'
)
_process_history(advanced_model_history)

**Your interpretation:**

_Compare the training and validation curves. Explain how augmentation, batch normalization, and dropout affected generalization._

## Transfer Learning Model

### **TASK: Transfer Learning with Pretrained ResNet50**

Use an ImageNet-pretrained ResNet50 as a frozen feature extractor:

1. Complete the training and test transforms. Resize images to 224 × 224 and normalize them with the supplied ImageNet statistics. Apply augmentation only to the training images.
2. Load ResNet50 with pretrained weights.
3. Freeze every backbone parameter.
4. Replace the original ResNet classification layer with an identity operation and add a new trainable classifier for the 10 CIFAR-10 classes.
5. Optimize only parameters that still require gradients.
6. Train and save the transfer-learning model.

In [ ]:
weights = ResNet50_Weights.DEFAULT
imagenet_mean = weights.transforms().mean
imagenet_std = weights.transforms().std

transfer_train_transform = transforms.Compose([
  # ---- YOUR CODE GOES HERE (START) ----
  # Add translation, horizontal flipping, rotation, and resize to 224 x 224.
  # ---- YOUR CODE GOES HERE (END) ----
  transforms.ToTensor(),
  transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])
transfer_test_transform = transforms.Compose([
  # ---- YOUR CODE GOES HERE (START) ----
  # Resize to 224 x 224.
  # ---- YOUR CODE GOES HERE (END) ----
  transforms.ToTensor(),
  transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

transfer_train_data = datasets.CIFAR10(
  root=DATA_DIR, train=True, download=False,
  transform=transfer_train_transform
)
transfer_test_data = datasets.CIFAR10(
  root=DATA_DIR, train=False, download=False,
  transform=transfer_test_transform
)

In [ ]:
class TransferResNet50(nn.Module):
  def __init__(self, pretrained=True):
    super().__init__()
    selected_weights = ResNet50_Weights.DEFAULT if pretrained else None
    self.backbone = resnet50(weights=selected_weights)
    input_features = self.backbone.fc.in_features

    # ---- YOUR CODE GOES HERE (START) ----
    # Freeze the backbone and replace its original classification layer.
    # ---- YOUR CODE GOES HERE (END) ----

    self.classifier = nn.Linear(input_features, 10)

  def train(self, mode=True):
    super().train(mode)
    # Keep frozen BatchNorm statistics unchanged during head training.
    self.backbone.eval()
    return self

  def forward(self, inputs):
    features = self.backbone(inputs)
    return self.classifier(features)


def build_transfer_model(pretrained=True):
  return TransferResNet50(pretrained=pretrained)

In [ ]:
BATCH_SIZE = 64
EPOCHS = 20

transfer_train_loader = DataLoader(
    transfer_train_data, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT_WORKERS
)
transfer_test_loader = DataLoader(
    transfer_test_data, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT_WORKERS
)

transfer_model = build_transfer_model(pretrained=True).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    (parameter for parameter in transfer_model.parameters()
     if parameter.requires_grad),
    lr=1e-3,
)

transfer_model_history = _train_model(
    transfer_model, transfer_train_loader, transfer_test_loader,
    loss_fn, optimizer, EPOCHS
)
_save_trained_model(
    transfer_model, transfer_model_history, 'transfer_model_pytorch'
)

### **TASK: Training Curve**

Plot the training curve and interpret it: Was the training successful? Did it stop at the right time?

In [ ]:
transfer_model_history, transfer_model = _load_trained_model(
  build_transfer_model(pretrained=False), 'transfer_model_pytorch'
)
_process_history(transfer_model_history)

**Your interpretation:**

_Explain whether transfer learning converged faster and whether it improved validation performance compared with the two models trained from scratch._

## Performance Comparison

### **TASK: Performance Comparison**

Compare the three models to analyze how model complexity, regularization, augmentation, and pretrained knowledge affect their training dynamics and validation performance.

In [ ]:
def _compare_models(model_histories):
  # ---- YOUR CODE GOES HERE (START) ----
  # Create a 2 x 2 figure
  fig, axes = None, None
  # ---- YOUR CODE GOES HERE (END) ----


  colors = plt.cm.tab10.colors
  plots = [
    ('loss', 'Training Loss', axes[0, 0]),
    ('val_loss', 'Validation Loss', axes[0, 1]),
    ('accuracy', 'Training Accuracy', axes[1, 0]),
    ('val_accuracy', 'Validation Accuracy', axes[1, 1]),
  ]

  for metric, title, axis in plots:
    # ---- YOUR CODE GOES HERE (START) ----
    # Plot the metric for each model in model_histories on the given axis
    # ---- YOUR CODE GOES HERE (END) ----
    axis.set_title(title)
    axis.legend()
    axis.grid(True, alpha=0.3)

  axes[0, 0].set_ylabel('Loss')
  axes[1, 0].set_ylabel('Accuracy')
  axes[1, 0].set_xlabel('Epoch')
  axes[1, 1].set_xlabel('Epoch')
  plt.tight_layout()
  plt.show()


model_histories = {
  'Base Model': base_model_history,
  'Advanced Model': advanced_model_history,
  'Transfer Model': transfer_model_history,
}

_compare_models(model_histories)